# Ollama + GDELT Integration Demo

This notebook demonstrates how to use Ollama with various AI models for GDELT data analysis. Ollama provides an easy way to run large language models locally with a simple API.

## Overview

- **Ollama**: Simple, fast way to run LLMs locally
- **Model Choice**: Support for multiple models (Llama, Mistral, CodeLlama, etc.)
- **GDELT Integration**: AI-powered analysis of global events
- **Comparison**: Alternative to OpenLLM with different trade-offs

## Advantages of Ollama vs OpenLLM

### Ollama Benefits:
- **Simpler Setup**: Single binary installation
- **Model Management**: Easy model download and switching
- **Resource Efficient**: Better memory management
- **Wide Model Support**: Llama 3.3, Mistral, Qwen, etc.
- **Fast Switching**: Quick model changes without restart

### OpenLLM Benefits:
- **Production Ready**: Built for deployment
- **OpenAI Compatible**: Drop-in API replacement
- **Advanced Features**: More configuration options

## Prerequisites

1. **Install Ollama**: Download from https://ollama.ai/
2. **Pull a model**: `ollama pull llama3.3:8b` (or your preferred model)
3. **Start Ollama**: Usually runs automatically after installation

Let's get started!

## Instructions for Running the Demo

To run this Ollama + GDELT integration demo:

### 1. Install and Start Ollama

Download and install Ollama from https://ollama.ai/, then:
```bash
# Pull a recommended model
ollama pull llama3.3:8b

# Or try other models:
ollama pull mistral:7b
ollama pull qwen2.5:7b
```

Ollama runs automatically after installation (no manual server start needed).

### 2. Run the Notebook Cells

Execute each cell in sequence. The notebook will:

1. **Setup**: Check connections to Ollama and GDELT database
2. **Demo 1**: Analyze recent global events with geopolitical insights
3. **Demo 2**: Focus on a specific country (US by default)
4. **Demo 3**: Analyze trends and provide predictions
5. **Demo 4**: Interactive analysis where you can ask custom questions

### 3. Expected Results

- **Geopolitical Analysis**: AI provides detailed analysis of global events
- **Country-Specific Insights**: Focused analysis on national activities and relationships
- **Trend Analysis**: Pattern recognition and future predictions
- **Interactive Q&A**: Real-time answers to your questions about the data

### 4. Performance Notes

- **Model Loading**: First query may take 30-60 seconds
- **Subsequent Queries**: 10-30 seconds per response
- **Resource Requirements**: Much lighter than DeepSeek (4-8GB RAM typically)
- **CPU Performance**: Works well on CPU, GPU optional

### 5. Integration Points

This demo showcases how to:
- Combine AI reasoning with real-time data using simpler setup
- Generate actionable intelligence from raw events
- Provide context-aware analysis with local models
- Create interactive data exploration tools

The integration can be extended to:
- Generate automated reports
- Create early warning systems
- Provide real-time briefings
- Support decision-making processes

In [ ]:
# Import required libraries
import ollama
import pandas as pd
import duckdb
import time
import json
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import numpy as np

print("Setting up Ollama + GDELT Integration...")

# Configuration
GDELT_DB_PATH = "../../etl/duckdatabase"
DEFAULT_MODEL = "llama3.3:8b"  # You can change this to any model you have installed
DEFAULT_MODEL = "llama3.2:latest"
def check_ollama_status():
    """Check if Ollama is running and what models are available"""
    try:
        # List available models
        models = ollama.list()
        return True, models
    except Exception as e:
        return False, str(e)

def query_ollama(prompt, model_name=DEFAULT_MODEL, temperature=0.7, max_tokens=1000):
    """Send a query to Ollama with the specified model"""
    try:
        response = ollama.chat(
            model=model_name,
            messages=[{
                'role': 'user',
                'content': prompt
            }],
            options={
                'temperature': temperature,
                'num_predict': max_tokens,
            }
        )
        return response
    except Exception as e:
        return {"error": f"Ollama request failed: {e}"}

# Check Ollama status
print("🔍 Checking Ollama status...")
is_running, models_info = check_ollama_status()

if is_running:
    print("✅ Ollama is running!")
    print(f"📦 Available models: {len(models_info['models'])} models")
    
    # Display available models
    print("\nAvailable Models:")
    print(models_info)
    for model in models_info['models']:
        name = model['model']
        size = model.get('size', 0) / (1024**3)  # Convert to GB
        print(f"  - {name} ({size:.1f}GB)")
    
    # Check if default model is available
    model_names = [m['model'] for m in models_info['models']]
    if DEFAULT_MODEL not in model_names:
        print(f"\n⚠️  Default model '{DEFAULT_MODEL}' not found!")
        print("Available alternatives:", model_names[:3])
        if model_names:
            DEFAULT_MODEL = model_names[0]
            print(f"Using '{DEFAULT_MODEL}' instead")
    else:
        print(f"✅ Using model: {DEFAULT_MODEL}")
        
else:
    print("❌ Ollama not accessible. Please:")
    print("1. Install Ollama from https://ollama.ai/")
    print("2. Run: ollama pull llama3.3:8b")
    print("3. Ensure Ollama service is running")

# Test GDELT database connection
def test_gdelt_connection():
    """Test connection to GDELT database"""
    try:
        conn = duckdb.connect(GDELT_DB_PATH)
        result = conn.execute("SELECT COUNT(*) FROM gdelt_events").fetchone()
        event_count = result[0] if result else 0
        conn.close()
        return event_count
    except Exception as e:
        print(f"GDELT DB Error: {e}")
        return 0

event_count = test_gdelt_connection()
if event_count > 0:
    print(f"✅ GDELT database connected! Found {event_count:,} events")
else:
    print("❌ GDELT database not accessible. Check path:", GDELT_DB_PATH)

## Demo 1: GDELT Data Overview Analysis

Let's ask Ollama to analyze a sample of GDELT data and provide insights about global events patterns.

In [ ]:
# Fetch recent GDELT events for analysis
def get_recent_gdelt_sample(limit=50):
    """Get recent GDELT events for analysis"""
    try:
        conn = duckdb.connect(GDELT_DB_PATH)
        
        query = """
        SELECT 
            GLOBALEVENTID,
            SQLDATE,
            Actor1Name,
            Actor2Name,
            EventCode,
            GoldsteinScale,
            AvgTone,
            ActionGeo_CountryCode,
            ActionGeo_Fullname
        FROM gdelt_events 
        WHERE SQLDATE >= (SELECT MAX(SQLDATE) - 7 FROM gdelt_events)
        AND Actor1Name IS NOT NULL 
        AND Actor2Name IS NOT NULL
        ORDER BY SQLDATE DESC 
        LIMIT ?
        """
        
        df = conn.execute(query, [limit]).df()
        conn.close()
        return df
    except Exception as e:
        print(f"Error fetching GDELT data: {e}")
        # Create sample data for demo
        sample_data = {
            'GLOBALEVENTID': range(1001, 1021),
            'SQLDATE': [20250129] * 10 + [20250128] * 10,
            'Actor1Name': ['UNITED STATES', 'CHINA', 'EUROPEAN UNION', 'RUSSIA', 'JAPAN'] * 4,
            'Actor2Name': ['CHINA', 'UNITED STATES', 'RUSSIA', 'UKRAINE', 'SOUTH KOREA'] * 4,
            'EventCode': ['010', '051', '043', '020', '036'] * 4,
            'GoldsteinScale': [-2.0, -5.0, 3.0, -8.0, 4.0] * 4,
            'AvgTone': [-1.5, -4.2, 2.1, -6.8, 3.2] * 4,
            'ActionGeo_CountryCode': ['US', 'CH', 'GM', 'UP', 'JA'] * 4,
            'ActionGeo_Fullname': ['Washington, DC', 'Beijing', 'Berlin', 'Kiev', 'Tokyo'] * 4
        }
        return pd.DataFrame(sample_data)

# Get sample data
print("Fetching recent GDELT events...")
gdelt_sample = get_recent_gdelt_sample(20)

if not gdelt_sample.empty:
    print(f"✅ Fetched {len(gdelt_sample)} recent events")
    print("\nSample of recent events:")
    display(gdelt_sample.head())
    
    # Prepare summary for AI analysis
    event_summary = []
    for _, row in gdelt_sample.head(10).iterrows():
        event_summary.append({
            'actors': f"{row['Actor1Name']} -> {row['Actor2Name']}",
            'event_code': row['EventCode'],
            'goldstein': row['GoldsteinScale'],
            'tone': row['AvgTone'],
            'location': row['ActionGeo_Fullname'],
            'country': row['ActionGeo_CountryCode']
        })
    
    # Convert to readable format for AI
    summary_text = "Recent GDELT Events Summary:\n\n"
    for i, event in enumerate(event_summary, 1):
        summary_text += f"{i}. {event['actors']} (Event: {event['event_code']}, "
        summary_text += f"Goldstein: {event['goldstein']}, Tone: {event['tone']:.1f}, "
        summary_text += f"Location: {event['location']})\n"
    
    print("\n" + "="*60)
    print("PREPARED SUMMARY FOR AI ANALYSIS:")
    print("="*60)
    print(summary_text)
    
else:
    print("❌ No GDELT data available for analysis")

In [ ]:
# Ask Ollama to analyze the GDELT events
if 'summary_text' in locals() and is_running:
    analysis_prompt = f"""
    As a geopolitical analyst, please analyze these recent GDELT events and provide insights:

    {summary_text}

    Please provide:
    1. Key patterns you observe in these events
    2. Notable geopolitical trends or tensions
    3. Countries or regions showing significant activity
    4. Event types that are most prevalent
    5. Overall tone analysis (positive/negative trends)
    6. Any potential implications or connections between events

    Focus on actionable insights that would be valuable for monitoring global events.
    """
    
    print("🤖 Asking Ollama to analyze the GDELT events...")
    print("This may take 30-60 seconds...\n")
    
    start_time = time.time()
    response = query_ollama(analysis_prompt, DEFAULT_MODEL, temperature=0.3, max_tokens=1500)
    end_time = time.time()
    
    if "error" not in response:
        analysis = response['message']['content']
        print("🔍 OLLAMA GEOPOLITICAL ANALYSIS:")
        print("="*80)
        print(analysis)
        print("="*80)
        print(f"\n⏱️ Analysis completed in {end_time - start_time:.1f} seconds")
        
        # Store the analysis for later use
        ollama_analysis = analysis
        
    else:
        print(f"❌ Error: {response['error']}")
        
else:
    print("⚠️ Skipping AI analysis - either no data or server not available")

## Demo 2: Country-Specific Analysis

Let's focus on a specific country and ask Ollama to provide detailed analysis of events involving that country.

In [ ]:
# Focus on a specific country for detailed analysis
def analyze_country_events(country_code, days_back=7):
    """Get events for a specific country"""
    try:
        conn = duckdb.connect(GDELT_DB_PATH)
        
        # Calculate date range
        end_date = datetime.now()
        start_date = end_date - timedelta(days=days_back)
        start_date_str = start_date.strftime("%Y%m%d")
        
        query = """
        SELECT 
            GLOBALEVENTID,
            SQLDATE,
            Actor1Name,
            Actor2Name,
            EventCode,
            GoldsteinScale,
            AvgTone,
            ActionGeo_Fullname,
            ActionGeo_CountryCode
        FROM gdelt_events 
        WHERE (ActionGeo_CountryCode = ? OR Actor1CountryCode = ? OR Actor2CountryCode = ?)
        AND SQLDATE >= ?
        ORDER BY SQLDATE DESC, GoldsteinScale DESC
        LIMIT 15
        """
        
        df = conn.execute(query, [country_code, country_code, country_code, start_date_str]).df()
        conn.close()
        return df
    except Exception as e:
        print(f"Error fetching country data: {e}")
        # Create sample data for demo
        sample_data = {
            'GLOBALEVENTID': range(2001, 2011),
            'SQLDATE': [20250129] * 5 + [20250128] * 5,
            'Actor1Name': ['UNITED STATES'] * 10,
            'Actor2Name': ['CHINA', 'RUSSIA', 'EUROPEAN UNION', 'JAPAN', 'CANADA'] * 2,
            'EventCode': ['010', '051', '043', '020', '036'] * 2,
            'GoldsteinScale': [-2.0, -5.0, 3.0, -8.0, 4.0] * 2,
            'AvgTone': [-1.5, -4.2, 2.1, -6.8, 3.2] * 2,
            'ActionGeo_Fullname': ['Washington, DC', 'New York', 'Los Angeles', 'Chicago', 'Miami'] * 2,
            'ActionGeo_CountryCode': ['US'] * 10
        }
        return pd.DataFrame(sample_data)

# Choose a country to analyze (let's use US for this example)
target_country = "US"
print(f"🇺🇸 Analyzing recent events involving {target_country}...")

country_events = analyze_country_events(target_country, days_back=14)

if not country_events.empty:
    print(f"✅ Found {len(country_events)} events involving {target_country}")
    
    # Create summary for AI
    country_summary = f"Recent events involving {target_country} (last 14 days):\n\n"
    
    for i, (_, row) in enumerate(country_events.iterrows(), 1):
        country_summary += f"{i}. {row['Actor1Name']} -> {row['Actor2Name']} "
        country_summary += f"(Event: {row['EventCode']}, Goldstein: {row['GoldsteinScale']}, "
        country_summary += f"Tone: {row['AvgTone']:.1f}, Location: {row['ActionGeo_Fullname']})\n"
    
    print("📊 COUNTRY EVENTS SUMMARY:")
    print("="*60)
    print(country_summary)
    
    # Calculate basic statistics
    avg_goldstein = country_events['GoldsteinScale'].mean()
    avg_tone = country_events['AvgTone'].mean()
    
    print(f"\n📈 STATISTICS:")
    print(f"Average Goldstein Scale: {avg_goldstein:.2f}")
    print(f"Average Tone: {avg_tone:.2f}")
    print(f"Most Common Event Codes: {country_events['EventCode'].value_counts().head(3).to_dict()}")
    
else:
    print(f"❌ No recent events found for {target_country}")
    country_summary = None

In [ ]:
# Ask Ollama for country-specific analysis
if country_summary and is_running:
    country_prompt = f"""
    As a geopolitical analyst specializing in {target_country}, please analyze these recent events:

    {country_summary}

    Additional context:
    - Average Goldstein Scale: {avg_goldstein:.2f} (range: -10 to +10, where negative = conflict, positive = cooperation)
    - Average Tone: {avg_tone:.2f} (range: -100 to +100, where negative = negative sentiment)

    Please provide:
    1. Assessment of {target_country}'s current geopolitical position based on these events
    2. Key relationships and partnerships evident in the data
    3. Areas of tension or conflict involvement
    4. Dominant types of activities or engagements
    5. Sentiment analysis of {target_country}'s international interactions
    6. Potential implications for foreign policy and international relations
    7. Recommendations for monitoring or areas requiring attention

    Be specific and cite the events in your analysis.

    """
    
    print(f"🤖 Asking Ollama to analyze {target_country} events...")
    print("This may take 30-60 seconds...\n")
    
    start_time = time.time()
    response = query_ollama(country_prompt, DEFAULT_MODEL, temperature=0.3, max_tokens=2000)
    end_time = time.time()
    
    if "error" not in response:
        country_analysis = response['message']['content']
        print(f"🔍 OLLAMA {target_country} ANALYSIS:")
        print("="*80)
        print(country_analysis)
        print("="*80)
        print(f"\n⏱️ Analysis completed in {end_time - start_time:.1f} seconds")
        
    else:
        print(f"❌ Error: {response['error']}")
        
else:
    print("⚠️ Skipping country analysis - either no data or server not available")

## Demo 3: Trend Analysis and Predictions

Let's ask Ollama to analyze trends over time and provide predictions about future developments.

In [ ]:
# Analyze trends over time
def get_trend_data(days_back=30):
    """Get aggregated data for trend analysis"""
    try:
        conn = duckdb.connect(GDELT_DB_PATH)
        
        # Calculate date range
        end_date = datetime.now()
        start_date = end_date - timedelta(days=days_back)
        start_date_str = start_date.strftime("%Y%m%d")
        
        query = """
        SELECT 
            SQLDATE,
            COUNT(*) as event_count,
            AVG(GoldsteinScale) as avg_goldstein,
            AVG(AvgTone) as avg_tone,
            COUNT(DISTINCT ActionGeo_CountryCode) as countries_involved,
            ActionGeo_CountryCode as country,
            COUNT(*) as country_events
        FROM gdelt_events 
        WHERE SQLDATE >= ?
        AND GoldsteinScale IS NOT NULL
        AND AvgTone IS NOT NULL
        GROUP BY SQLDATE, ActionGeo_CountryCode
        ORDER BY SQLDATE DESC, country_events DESC
        LIMIT 100
        """
        
        df = conn.execute(query, [start_date_str]).df()
        conn.close()
        return df
    except Exception as e:
        print(f"Error fetching trend data: {e}")
        # Create sample trend data
        dates = [20250129, 20250128, 20250127, 20250126, 20250125]
        countries = ['US', 'CH', 'GM', 'UP', 'JA']
        sample_data = []
        for date in dates:
            for country in countries:
                sample_data.append({
                    'SQLDATE': date,
                    'event_count': np.random.randint(5, 25),
                    'avg_goldstein': np.random.uniform(-5, 5),
                    'avg_tone': np.random.uniform(-10, 10),
                    'countries_involved': len(countries),
                    'country': country,
                    'country_events': np.random.randint(5, 25)
                })
        return pd.DataFrame(sample_data)

print("📈 Fetching trend data for the last 30 days...")
trend_data = get_trend_data(30)

if not trend_data.empty:
    print(f"✅ Found trend data for {len(trend_data)} country-day combinations")
    
    # Aggregate by date
    daily_trends = trend_data.groupby('SQLDATE').agg({
        'event_count': 'sum',
        'avg_goldstein': 'mean',
        'avg_tone': 'mean',
        'country': 'count'  # This gives us countries per day
    }).reset_index()
    
    print("\n📊 DAILY TRENDS SUMMARY:")
    print("="*60)
    for _, row in daily_trends.head(10).iterrows():
        date_str = str(row['SQLDATE'])
        formatted_date = f"{date_str[:4]}-{date_str[4:6]}-{date_str[6:8]}"
        print(f"{formatted_date}: {row['event_count']} events, "
              f"Goldstein: {row['avg_goldstein']:.2f}, "
              f"Tone: {row['avg_tone']:.2f}, "
              f"Countries: {row['country']}")
    
    # Get most active countries
    country_activity = trend_data.groupby('country').agg({
        'country_events': 'sum',
        'avg_goldstein': 'mean',
        'avg_tone': 'mean'
    }).sort_values('country_events', ascending=False).head(10)
    
    print(f"\n🌍 TOP 10 MOST ACTIVE COUNTRIES:")
    print("="*60)
    for country, data in country_activity.iterrows():
        if pd.notna(country):  # Skip null countries
            print(f"{country}: {data['country_events']} events, "
                  f"Avg Goldstein: {data['avg_goldstein']:.2f}, "
                  f"Avg Tone: {data['avg_tone']:.2f}")
    
    # Prepare trend summary for AI
    trend_summary = f"""
GLOBAL EVENTS TREND ANALYSIS (Last 30 days):

Daily Activity Patterns:
"""
    for _, row in daily_trends.head(7).iterrows():
        date_str = str(row['SQLDATE'])
        formatted_date = f"{date_str[:4]}-{date_str[4:6]}-{date_str[6:8]}"
        trend_summary += f"- {formatted_date}: {row['event_count']} events (Goldstein: {row['avg_goldstein']:.2f}, Tone: {row['avg_tone']:.2f})\n"
    
    trend_summary += f"\nMost Active Countries (by event count):\n"
    for country, data in country_activity.head(5).iterrows():
        if pd.notna(country):
            trend_summary += f"- {country}: {data['country_events']} events (Goldstein: {data['avg_goldstein']:.2f}, Tone: {data['avg_tone']:.2f})\n"
    
    # Calculate overall statistics
    overall_goldstein = trend_data['avg_goldstein'].mean()
    overall_tone = trend_data['avg_tone'].mean()
    total_events = trend_data['country_events'].sum()
    
    trend_summary += f"""
Overall Statistics:
- Total Events: {total_events}
- Average Goldstein Scale: {overall_goldstein:.2f}
- Average Tone: {overall_tone:.2f}
- Unique Countries Involved: {trend_data['country'].nunique()}
"""
    
    print(f"\n🔍 TREND SUMMARY FOR AI:")
    print("="*80)
    print(trend_summary)
    
else:
    print("❌ No trend data available")
    trend_summary = None

In [ ]:
# Ask Ollama for trend analysis and predictions
if trend_summary and is_running:
    prediction_prompt = f"""
    As a geopolitical forecasting analyst, please analyze these global events trends and provide insights:

    {trend_summary}

    Please provide:
    1. Key trends and patterns you observe in the data
    2. Countries or regions showing concerning developments
    3. Areas of increased cooperation vs. conflict
    4. Sentiment shifts over time (based on tone analysis)
    5. Potential early warning signals or indicators to monitor
    6. Short-term predictions (next 1-2 weeks) based on these patterns
    7. Recommended focus areas for continued monitoring
    8. Any seasonal or cyclical patterns you notice

    Be specific about what the data suggests and provide actionable intelligence insights.
    Focus on identifying potential escalations, de-escalations, or significant diplomatic developments.
    """
    
    print("🤖 Asking Ollama for trend analysis and predictions...")
    print("This may take 60-90 seconds for comprehensive analysis...\n")
    
    start_time = time.time()
    response = query_ollama(prediction_prompt, DEFAULT_MODEL, temperature=0.4, max_tokens=2500)
    end_time = time.time()
    
    if "error" not in response:
        prediction_analysis = response['message']['content']
        print("🔮 OLLAMA TREND ANALYSIS & PREDICTIONS:")
        print("="*80)
        print(prediction_analysis)
        print("="*80)
        print(f"\n⏱️ Analysis completed in {end_time - start_time:.1f} seconds")
        
    else:
        print(f"❌ Error: {response['error']}")
        
else:
    print("⚠️ Skipping trend analysis - either no data or server not available")

## Demo 4: Interactive Analysis

Create an interactive function where you can ask Ollama specific questions about the GDELT data.

In [ ]:
# Interactive GDELT analysis with Ollama
def interactive_gdelt_analysis():
    """Interactive function to ask questions about GDELT data"""
    
    def get_custom_data(query_description):
        """Get data based on user description"""
        try:
            conn = duckdb.connect(GDELT_DB_PATH)
            
            # For demo, we'll provide some common query patterns
            if "conflict" in query_description.lower() or "negative" in query_description.lower():
                query = """
                SELECT Actor1Name, Actor2Name, EventCode, GoldsteinScale, AvgTone, 
                       ActionGeo_Fullname, ActionGeo_CountryCode
                FROM gdelt_events 
                WHERE GoldsteinScale < -5 AND AvgTone < -5
                ORDER BY GoldsteinScale ASC, AvgTone ASC
                LIMIT 10
                """
            elif "cooperation" in query_description.lower() or "positive" in query_description.lower():
                query = """
                SELECT Actor1Name, Actor2Name, EventCode, GoldsteinScale, AvgTone, 
                       ActionGeo_Fullname, ActionGeo_CountryCode
                FROM gdelt_events 
                WHERE GoldsteinScale > 5 AND AvgTone > 5
                ORDER BY GoldsteinScale DESC, AvgTone DESC
                LIMIT 10
                """
            else:
                # Default: recent high-impact events
                query = """
                SELECT Actor1Name, Actor2Name, EventCode, GoldsteinScale, AvgTone, 
                       ActionGeo_Fullname, ActionGeo_CountryCode
                FROM gdelt_events 
                WHERE SQLDATE >= (SELECT MAX(SQLDATE) - 7 FROM gdelt_events)
                ORDER BY ABS(GoldsteinScale) DESC
                LIMIT 10
                """
            
            df = conn.execute(query).df()
            conn.close()
            return df
        except Exception as e:
            print(f"Error fetching custom data: {e}")
            return pd.DataFrame()
    
    print("🔍 INTERACTIVE GDELT ANALYSIS WITH OLLAMA")
    print("="*60)
    print("Ask me questions about global events! Type 'quit' to exit.")
    print("\nExample questions:")
    print("- 'What conflicts are happening recently?'")
    print("- 'Show me positive cooperation events'")
    print("- 'What's happening with China?'")
    print("- 'Analyze recent diplomatic activities'")
    print()
    
    while True:
        try:
            user_question = input("\n🤔 Your question: ").strip()
            
            if user_question.lower() in ['quit', 'exit', 'bye', 'stop']:
                print("👋 Goodbye!")
                break
                
            if not user_question:
                continue
            
            print(f"\n🔄 Analyzing your question: '{user_question}'")
            
            # Get relevant data
            relevant_data = get_custom_data(user_question)
            
            if relevant_data.empty:
                print("❌ No relevant data found for your question.")
                continue
            
            # Format data for AI
            data_summary = f"Relevant GDELT events for: '{user_question}'\n\n"
            for i, (_, row) in enumerate(relevant_data.iterrows(), 1):
                data_summary += f"{i}. {row['Actor1Name']} -> {row['Actor2Name']} "
                data_summary += f"(Event: {row['EventCode']}, Goldstein: {row['GoldsteinScale']}, "
                data_summary += f"Tone: {row['AvgTone']:.1f}, Location: {row['ActionGeo_Fullname']})\n"
            
            # Create analysis prompt
            analysis_prompt = f'''
            User Question: "{user_question}"
            
            Relevant GDELT Data:
            {data_summary}
            
            Please provide a comprehensive answer to the user's question based on this GDELT data. 
            Be specific, cite the events, and provide actionable insights. 
            If the question asks for analysis, provide detailed geopolitical context.
            '''
            
            if not is_running:
                print("❌ Ollama server not available. Please start Ollama first.")
                continue
            
            print("🤖 Ollama is analyzing...")
            
            start_time = time.time()
            response = query_ollama(analysis_prompt, DEFAULT_MODEL, temperature=0.5, max_tokens=1500)
            end_time = time.time()
            
            if "error" not in response:
                answer = response['message']['content']
                print("\n🎯 OLLAMA ANALYSIS:")
                print("-" * 50)
                print(answer)
                print("-" * 50)
                print(f"⏱️ Response time: {end_time - start_time:.1f} seconds")
            else:
                print(f"❌ Error: {response['error']}")
                continue
            
        except KeyboardInterrupt:
            print("\n👋 Goodbye!")
            break
        except Exception as e:
            print(f"❌ Error: {e}")

# Example questions to try (uncomment to use interactively)
print("📝 SAMPLE QUESTIONS TO TRY:")
print("="*50)
sample_questions = [
    "What conflicts are happening recently?",
    "Show me positive cooperation events",
    "Analyze diplomatic activities",
    "What events show negative sentiment?",
    "What are the most significant recent events?"
]

for i, question in enumerate(sample_questions, 1):
    print(f"{i}. {question}")

print("\n💡 To start interactive analysis, uncomment the line below:")
print("# interactive_gdelt_analysis()")

# Uncomment the next line to start interactive mode
interactive_gdelt_analysis()

## Summary

This notebook demonstrated how to:

1. **Set up Ollama** for local AI model deployment
2. **Load and analyze GDELT data** from a DuckDB database
3. **Perform AI-powered analysis** of geopolitical events
4. **Compare different approaches** to local AI deployment
5. **Generate comprehensive reports** and insights

### Key Advantages of Ollama:

- **Privacy**: All processing happens locally
- **Cost-effective**: No API costs for inference
- **Customizable**: Easy to switch between different models
- **Fast**: Local processing eliminates network latency
- **Offline capable**: Works without internet connection
- **Simple setup**: Single binary installation vs complex OpenLLM setup

### Recommended Models for GDELT Analysis:

- **llama3.3:8b**: Good general-purpose model for most analyses
- **mistral:7b**: Excellent for reasoning and analytical tasks
- **qwen2.5:7b**: Strong performance with efficient resource usage
- **codellama:7b**: Good for data analysis and technical insights

### Next Steps:

1. Try different models and compare their analytical capabilities
2. Experiment with different temperature settings for varied responses
3. Create custom prompts for specific analytical needs
4. Integrate with real-time GDELT data feeds
5. Build automated monitoring and alerting systems
6. Compare results with the OpenLLM + DeepSeek approach

For more information about Ollama, visit: https://ollama.ai/